# 02. 데이터 전처리 및 피처 엔지니어링

EDA에서 발견한 인사이트를 바탕으로, 모델 학습에 적합한 데이터셋을 구축합니다.

## 핵심 설계: Data Leakage 방지

폐업 위험 예측은 **미래를 예측**하는 문제입니다. 따라서 예측 시점(t)에 알 수 없는 정보를 피처로 쓰면 안 됩니다.

- 🚫 **현재 시점(t)의 원본 데이터 사용 금지** — 예측 시점에는 아직 관측되지 않은 값
- 🚫 **폐업_률 / 폐업_점포_수 제외** — 타겟 그 자체 (직접 누수)
- ✅ **과거 시점(t-1, t-2)의 lag 피처만 사용**
- ✅ **그룹 통계량은 train 데이터에서만 계산** 후 val/test에 적용

## 처리 과정
1. 데이터 로드 및 범주형 인코딩
2. 타겟 변수 생성 (폐업률 → 이진 분류)
3. Lag 피처 생성 (t-1, t-2)
4. 파생 변수 생성 (변화율/수익성/경쟁 지표)
5. 시계열 기준 Train/Val/Test 분할
6. 그룹 통계량 (train-only) 및 저장

→ 최종 **66개 피처**

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

SEED = 42

## 1. 데이터 로드 및 범주형 인코딩

`00_data_merge.ipynb`에서 생성한 통합 데이터(137컬럼)를 불러옵니다.  
범주형 변수(자치구, 업종, 상권변화지표)는 고유값이 많아 **Label Encoding**을 적용합니다.

In [2]:
df = pd.read_csv('../data/processed/merged_data.csv', encoding='utf-8-sig')
print(f'원본 데이터: {df.shape}')

# 시간 변수 생성
df['year'] = df['기준_년분기_코드'].astype(str).str[:4].astype(int)
df['quarter'] = df['기준_년분기_코드'].astype(str).str[4:].astype(int)
df['year_quarter'] = df['year'] * 10 + df['quarter']

# 범주형 인코딩
le_district = LabelEncoder()
df['자치구_encoded'] = le_district.fit_transform(df['자치구_코드_명'])

le_industry = LabelEncoder()
df['업종_encoded'] = le_industry.fit_transform(df['서비스_업종_코드_명'])

le_change = LabelEncoder()
df['상권변화_encoded'] = le_change.fit_transform(df['상권_변화_지표'])

print(f'자치구: {len(le_district.classes_)}개, '
      f'업종: {len(le_industry.classes_)}개, '
      f'상권변화: {len(le_change.classes_)}개')

# 시계열 정렬 (그룹별 lag 생성을 위해)
df = df.sort_values(['자치구_encoded', '업종_encoded', 'year_quarter']).reset_index(drop=True)

원본 데이터: (39975, 137)
자치구: 25개, 업종: 63개, 상권변화: 4개


## 2. 타겟 변수 생성

### 왜 이진 분류인가?
- EDA에서 확인했듯 폐업률은 우측으로 치우친 분포 → 회귀는 일반화가 어려움 (`03_regression_trial` 참고)
- **"몇 %인지"보다 "위험한지 아닌지"**가 비즈니스적으로 더 유용

### 기준
- 폐업률 **중앙값 이상** → `closure_risk = 1` (위험)
- 폐업률 중앙값 미만 → `closure_risk = 0` (정상)

In [3]:
threshold = df['폐업_률'].quantile(0.5)
df['closure_risk'] = (df['폐업_률'] >= threshold).astype(int)

print(f'폐업률 임계값 (중앙값): {threshold:.2f}%')
print('클래스 분포:')
print(df['closure_risk'].value_counts())

폐업률 임계값 (중앙값): 2.30%
클래스 분포:
closure_risk
1    20320
0    19655
Name: count, dtype: int64


## 3. Lag 피처 생성 (t-1, t-2)

각 (자치구 x 업종) 그룹 내에서 직전 1~2분기의 값을 피처로 만듭니다.  
🚫 `폐업_률`, `폐업_점포_수`는 lag 대상에서 **제외** (타겟 누수 방지)

In [4]:
lag_cols = [
    '당월_매출_금액', '당월_매출_건수', '점포_수', '유사_업종_점포_수',
    '프랜차이즈_점포_수', '개업_률', '개업_점포_수',
    '전체임대료', '총_유동인구_수', '총_상주인구_수', '총_직장인구_수',
    '토요일_매출_금액', '일요일_매출_금액', '시간대_21_24_매출_금액',
    '연령대_10_매출_금액', '연령대_20_매출_금액', '연령대_30_매출_금액',
    '연령대_40_매출_금액', '연령대_50_매출_금액', '연령대_60_이상_매출_금액'
]

group = df.groupby(['자치구_encoded', '업종_encoded'])
for col in lag_cols:
    df[f'{col}_lag1'] = group[col].shift(1)
    df[f'{col}_lag2'] = group[col].shift(2)

print(f'Lag 피처 {len(lag_cols) * 2}개 생성 (대상 {len(lag_cols)}개 컬럼 x 2)')

Lag 피처 40개 생성 (대상 20개 컬럼 x 2)


## 4. 파생 변수 생성 (모두 t-1, t-2 기반)

비즈니스 의미가 있는 파생 피처를 생성합니다. 모든 계산은 lag 값만 사용합니다.

In [5]:
g = df.groupby(['자치구_encoded', '업종_encoded'])

# 변화율 (t-1 대비 t-2)
df['매출_변화율'] = (df['당월_매출_금액_lag1'] - df['당월_매출_금액_lag2']) / (df['당월_매출_금액_lag2'] + 1)
df['매출건수_변화율'] = (df['당월_매출_건수_lag1'] - df['당월_매출_건수_lag2']) / (df['당월_매출_건수_lag2'] + 1)
df['점포수_변화율'] = (df['점포_수_lag1'] - df['점포_수_lag2']) / (df['점포_수_lag2'] + 1)
df['개업률_변화'] = df['개업_률_lag1'] - df['개업_률_lag2']

# 연속 하락 추세
df['매출_감소'] = (df['매출_변화율'] < 0).astype(int)
df['연속_매출_감소'] = g['매출_감소'].shift(1).rolling(2, min_periods=1).sum()

# 과거 추세
df['매출_3분기_평균'] = g['당월_매출_금액_lag1'].shift(1).rolling(3, min_periods=1).mean()
df['매출_추세_대비'] = df['당월_매출_금액_lag1'] / (df['매출_3분기_평균'] + 1)

# 수익성 지표 (t-1)
df['점포당_매출'] = df['당월_매출_금액_lag1'] / (df['점포_수_lag1'] + 1)
df['건당_매출'] = df['당월_매출_금액_lag1'] / (df['당월_매출_건수_lag1'] + 1)
df['임대료_부담률'] = df['전체임대료_lag1'] / (df['당월_매출_금액_lag1'] + 1)
df['유동인구_전환율'] = df['당월_매출_건수_lag1'] / (df['총_유동인구_수_lag1'] + 1)

# 고객 구조 (t-1)
age_cols_lag1 = [f'연령대_{age}_매출_금액_lag1' for age in ['10', '20', '30', '40', '50', '60_이상']]
df['최대_연령대_비중'] = df[age_cols_lag1].max(axis=1) / (df['당월_매출_금액_lag1'] + 1)
df['주말_매출_비율'] = (df['토요일_매출_금액_lag1'] + df['일요일_매출_금액_lag1']) / (df['당월_매출_금액_lag1'] + 1)
df['야간_매출_비율'] = df['시간대_21_24_매출_금액_lag1'] / (df['당월_매출_금액_lag1'] + 1)

# 경쟁 환경 (t-1)
df['프랜차이즈_비율'] = df['프랜차이즈_점포_수_lag1'] / (df['점포_수_lag1'] + 1)
df['경쟁_밀도'] = df['유사_업종_점포_수_lag1'] / (df['점포_수_lag1'] + 1)
df['점포_포화도'] = df['점포_수_lag1'] / ((df['총_상주인구_수_lag1'] + df['총_직장인구_수_lag1']) + 1) * 10000

# 무한대/결측 처리
df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
print('파생 변수 생성 완료')

파생 변수 생성 완료


## 5. 피처 선택 및 시계열 분할

### 피처 선택 원칙
- 현재 시점(t)의 모든 원본 컬럼 제외 → **lag/파생 피처만 사용**
- 타겟(`폐업_률`, `closure_risk`)과 식별/시간 컬럼 제외

### 분할 방식
미래 예측 문제이므로 **시간 순서**로 분할합니다 (랜덤 분할 금지).  
각 그룹의 초기 2개 분기는 lag2가 없어 제거합니다.

In [6]:
categorical_features = ['자치구_encoded', '업종_encoded', '상권변화_encoded']

# 원본 시점(t) 컬럼 + 타겟/식별/시간 컬럼 → 피처에서 제외
# (lag/파생이 아닌 merged 원본 수치·범주 컬럼 전부)
raw_value_cols = [c for c in df.columns
                  if not c.endswith('_lag1') and not c.endswith('_lag2')]

derived_features = [
    '매출_변화율', '매출건수_변화율', '점포수_변화율', '개업률_변화',
    '매출_감소', '연속_매출_감소', '매출_3분기_평균', '매출_추세_대비',
    '점포당_매출', '건당_매출', '임대료_부담률', '유동인구_전환율',
    '최대_연령대_비중', '주말_매출_비율', '야간_매출_비율',
    '프랜차이즈_비율', '경쟁_밀도', '점포_포화도'
]

# lag 피처 (생성 순서 유지)
lag_features = []
for col in lag_cols:
    lag_features += [f'{col}_lag1', f'{col}_lag2']

# 최종 피처 = 범주형 3 + lag 40 + 파생 18 (그룹통계 5는 6단계에서 추가)
feature_cols = categorical_features + lag_features + derived_features
print(f'현재 피처 수: {len(feature_cols)} (그룹통계 추가 전)')

현재 피처 수: 61 (그룹통계 추가 전)


In [7]:
# 각 그룹의 초기 2개 분기 제거 (lag2가 존재하지 않음)
# 단, 그룹 크기가 2개 이하이면 그대로 유지 (제거 시 그룹이 사라지는 것 방지)
df = df.sort_values(['자치구_encoded', '업종_encoded', 'year_quarter']).reset_index(drop=True)
grp = df.groupby(['자치구_encoded', '업종_encoded'])
df['_grp_idx'] = grp.cumcount()
df['_grp_size'] = grp['year_quarter'].transform('size')
df_clean = df[(df['_grp_idx'] >= 2) | (df['_grp_size'] <= 2)]
df_clean = df_clean.drop(columns=['_grp_idx', '_grp_size']).reset_index(drop=True)

# 시계열 정렬 후 시간 순서로 분할
df_sorted = df_clean.sort_values(
    ['year_quarter', '자치구_encoded', '업종_encoded']
).reset_index(drop=True)

n_total = len(df_sorted)
n_train = int(n_total * 0.7)
n_val = int(n_total * 0.1)

train_df = df_sorted.iloc[:n_train].copy()
val_df = df_sorted.iloc[n_train:n_train + n_val].copy()
test_df = df_sorted.iloc[n_train + n_val:].copy()

print(f'전체: {n_total}')
print(f'Train: {len(train_df)} ({len(train_df)/n_total*100:.1f}%)')
print(f'Val:   {len(val_df)} ({len(val_df)/n_total*100:.1f}%)')
print(f'Test:  {len(test_df)} ({len(test_df)/n_total*100:.1f}%)')

전체: 36875
Train: 25812 (70.0%)
Val:   3687 (10.0%)
Test:  7376 (20.0%)


## 6. 그룹 통계량 (Train-only) 및 저장

상권/업종별 평균 매출은 **train 데이터에서만 계산**하고 val/test에 매핑합니다.  
→ val/test 정보가 train 피처 생성에 새어 들어가는 것을 방지 (Leakage 방지)

In [8]:
# train에서만 그룹 평균 계산
district_sales_mean = train_df.groupby('자치구_encoded')['당월_매출_금액_lag1'].mean()
industry_sales_mean = train_df.groupby('업종_encoded')['당월_매출_금액_lag1'].mean()

for dataset in [train_df, val_df, test_df]:
    dataset['상권_매출_평균'] = dataset['자치구_encoded'].map(district_sales_mean)
    dataset['업종_매출_평균'] = dataset['업종_encoded'].map(industry_sales_mean)
    dataset['상권_대비_매출'] = dataset['당월_매출_금액_lag1'] / (dataset['상권_매출_평균'] + 1)
    dataset['업종_대비_매출'] = dataset['당월_매출_금액_lag1'] / (dataset['업종_매출_평균'] + 1)
    dataset['위험_점수'] = (
        (dataset['매출_변화율'] < -0.1).astype(int) * 2 +
        (dataset['임대료_부담률'] > 0.3).astype(int) * 2 +
        (dataset['상권_대비_매출'] < 0.7).astype(int) +
        (dataset.get('연속_매출_감소', 0) >= 2).astype(int) * 3
    )

# 그룹 통계 피처 추가
group_stat_features = ['상권_매출_평균', '업종_매출_평균', '상권_대비_매출', '업종_대비_매출', '위험_점수']
feature_cols = feature_cols + group_stat_features

# 결측/무한대 정리
for dataset in [train_df, val_df, test_df]:
    dataset.replace([np.inf, -np.inf], np.nan, inplace=True)
    dataset.fillna(0, inplace=True)

print(f'최종 피처 수: {len(feature_cols)}')

최종 피처 수: 66


In [9]:
# feature_cols만 추출해 저장
train_df[feature_cols].to_csv('../data/processed/train_features.csv', index=False, encoding='utf-8-sig')
train_df[['closure_risk']].to_csv('../data/processed/train_target.csv', index=False, encoding='utf-8-sig')
val_df[feature_cols].to_csv('../data/processed/val_features.csv', index=False, encoding='utf-8-sig')
val_df[['closure_risk']].to_csv('../data/processed/val_target.csv', index=False, encoding='utf-8-sig')
test_df[feature_cols].to_csv('../data/processed/test_features.csv', index=False, encoding='utf-8-sig')
test_df[['closure_risk']].to_csv('../data/processed/test_target.csv', index=False, encoding='utf-8-sig')

print('저장 완료: ../data/processed/')
print(f'  train_features: {train_df[feature_cols].shape}')
print(f'  val_features:   {val_df[feature_cols].shape}')
print(f'  test_features:  {test_df[feature_cols].shape}')

저장 완료: ../data/processed/
  train_features: (25812, 66)
  val_features:   (3687, 66)
  test_features:  (7376, 66)


## 정리

### 전처리 파이프라인 요약

| 단계 | 내용 | 비고 |
|------|------|------|
| 1 | 범주형 Label Encoding | 자치구(25) / 업종(63) / 상권변화(4) |
| 2 | 타겟 이진화 | 폐업률 중앙값 기준 (>=) |
| 3 | Lag 피처 (40개) | t-1, t-2 (폐업 관련 제외) |
| 4 | 파생 변수 (18개) | 변화율/수익성/경쟁 지표 |
| 5 | 시계열 분할 | Train 70 / Val 10 / Test 20 |
| 6 | 그룹 통계 (5개) | train-only 계산 (Leakage 방지) |

**최종 피처: 66개** (범주형 3 + lag 40 + 파생 18 + 그룹통계 5)

→ 다음: `03_regression_trial.ipynb` — 회귀 모델 시도 및 분류 전환 근거